In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# NB_Config_MD_B1: the one config of the colocated Master Data Bronze (workspace SAP_DAB_BRZ_PROD)
#   Read by NB_Bronze_MD_B1 (Bronze2 load), NB_MD_B1_Swap (B1 swap), NB_MD_B1_Copy (optional backup)
#   and NB_Check_B1_Keys. Lean on purpose: Bronze settings and keys only. Columns, renames, joins and enrichment
#   belong to Silver, whose configs (NB_Config_MD) are unchanged.
#   Generated 23 Nov 2026 from NB_Config_MD (157 tables); edit by hand from here on.
# ════════════════════════════════════════════════════════════════════════════

# ── 1. Run settings ─────────────────────────────────────────────────────────────
# Variable Library VL_MD_Bronze2: B1 = Dab's lakehouses (read and swapped), B2 = ours (Bronze2 + every log).
# Every value is a lakehouse ABFS path ending in /Tables; tables are under dbo/.
PIPELINE   = 'MD_B1'
VL_NAME    = 'VL_MD_Bronze2'
B2_KEY     = 'B2_MD_LH_ABFSS'              # our Bronze2 lakehouse
LAKEHOUSES = {
    'bronze': B2_KEY,                 # Bronze2 tables
    'ops':    'bronze',               # run, table and DQ logs: same lakehouse (no separate OPS lakehouse)
}

# Bronze source: B1. A table's source_lh_key / source_schema override these.
# Two more B1 sources: EINA / EINE / EORD are in Dab's PROC lakehouse (B1_PROC_LH_ABFSS), CSKT in the audit one (B1_AUDIT_LH_ABFSS).
SOURCE_LH_KEY    = 'B1_MD_LH_ABFSS'
SOURCE_SCHEMA    = 'dbo'
BRONZE_MODE      = 'cdc'              # 'cdc' keeps history and soft deletes; 'overwrite' replaces the table
DESC_BRONZE_MODE = 'overwrite'        # description (text) tables

# ── 2. Swap (NB_MD_B1_Swap): Dab writes <table>_stg next to the live B1 table, the swap switches it in ──
# Safeguards (NB_B1_Swap): dry run by default; staging must be complete (min_rows_ratio), left alone for quiet_minutes and
# newer than live; no lost or retyped columns; rename, verify, roll back from <table>_prev on any failed check.
DEFAULTS = {
    'tables_folder':    'dbo',        # between the VL value (…/Tables) and the tables
    'stg_suffix':       '_stg',       # staging folder = <table>_stg, the previous copy = <table>_prev
    'prev_suffix':      '_prev',
    'min_rows_ratio':   1.0,          # staging needs at least this share of the live rows (0.98 = up to 2% smaller)
    'quiet_minutes':    10,           # staging must have been left alone this long (a load may still be running)
    'delete_prev':      True,         # delete the previous copy once the new table is verified
    'require_new_data': True,         # False: also swap staging not written after the live table (testing only)
    'dry_run':          True,         # True: decide and log only. Set False to start swapping
    'parallel_tables':  1,            # tables swapped at the same time (domain level only)
    'range':            None,         # range swap (SWAP_SCOPE = 'range'): {'year_col', 'period_col', 'periods'}
}
MAX_PARALLEL_DOMAINS = 1
# b1_swap_log goes to B2 (our lakehouse), never into Dab's B1 lakehouse
LOG_TABLE_PATH = notebookutils.variableLibrary.getLibrary(VL_NAME)[B2_KEY].strip().rstrip('/') + '/dbo/b1_swap_log'

SWAP_DOMAINS = {
    'MD': {
        'vl':     VL_NAME,
        'lh_key': SOURCE_LH_KEY,      # B1: the swap renames folders in Dab's lakehouse
        'parallel_tables': 4,
        'tables': [                        # 30 tables (user list, 23 Nov 2026)
        'LFA1', 'LFB1', 'LFBK', 'LFC1', 'LFM1', 'KNVV', 'KNB1', 'KNA1', 'ADCP', 'ADR2', 'ADR3', 'ADR6', 'ADRC',
        'ADRCT', 'ADRP', 'ADRT', 'KNBK', 'KNB5', 'KNKA', 'KNKK', 'KNVH', 'KNVK', 'KNVP', 'KONA', 'KNC1', 'KNMT',
        'ZSDINDUSTRY_SEG0', 'ZSDINDUSTRY_SEG1', 'ZSDINDUSTRY_SEG2', 'ZSDINDUSTRY_SEG3'
        ],
    },
}

# ── 3. Backup copy (NB_MD_B1_Copy, optional): not needed for the swap, which keeps its own rollback (<table>_prev) ──
# Empty = nothing to copy. To back up B1 first, add the backup lakehouse to the Variable Library (e.g. B1_MD_BACKUP_LH_ABFSS)
# and an entry like:
#   {'name': 'MD B1 backup', 'source_vl': VL_NAME, 'source_key': SOURCE_LH_KEY, 'source_folder': 'dbo',
#    'target_vl': VL_NAME, 'target_key': 'B1_MD_BACKUP_LH_ABFSS', 'target_folder': 'dbo', 'tables': '*'},
COPIES = []
COPY_WORKERS = 4
COPY_DRY_RUN = True                   # True: list what would be copied and change nothing
COPY_ONLY    = ''                     # blank = every entry; or the start of entry names

# ── 4. Tables ───────────────────────────────────────────────────────────────────
# One line per table: primary_keys (SAP key, without MANDT), and only what differs from the settings above.
# Description tables: text_col (checked in the source), spras (True = SPRAS is part of the key).
MD_TABLE_CONFIGS = {                  # 53 tables
    'MARA': {'primary_keys': ['MATNR']},
    'MARC': {'primary_keys': ['MATNR', 'WERKS']},
    'MARM': {'primary_keys': ['MATNR', 'MEINH']},
    'MVKE': {'primary_keys': ['MATNR', 'VKORG', 'VTWEG']},
    'MSTA': {'primary_keys': ['MATNR', 'STATM', 'ZHLER']},
    'KNA1': {'primary_keys': ['KUNNR']},
    'KNVV': {'primary_keys': ['KUNNR', 'VKORG', 'VTWEG', 'SPART']},
    'KNVP': {'primary_keys': ['KUNNR', 'VKORG', 'VTWEG', 'SPART', 'PARVW', 'PARZA']},
    'LFA1': {'primary_keys': ['LIFNR']},
    'LFB1': {'primary_keys': ['LIFNR', 'BUKRS']},
    'LFM1': {'primary_keys': ['LIFNR', 'EKORG']},
    'T001K': {'primary_keys': ['BWKEY']},
    'T001W': {'primary_keys': ['WERKS']},
    'T001': {'primary_keys': ['BUKRS']},
    'T024E': {'primary_keys': ['EKORG']},
    'T024': {'primary_keys': ['EKGRP']},
    'TVKO': {'primary_keys': ['VKORG']},
    'TKA02': {'primary_keys': ['KOKRS', 'BUKRS']},
    'MBEW': {'primary_keys': ['MATNR', 'BWKEY']},
    'TVRO': {'primary_keys': ['ROUTE']},
    'ADRC': {'primary_keys': ['ADDRNUMBER', 'DATE_FROM', 'NATION']},
    'ZMM_MAT_ORG': {'primary_keys': ['MATNR', 'VKORG']},
    'TFACS': {'primary_keys': ['IDENT', 'JAHR']},
    'TFACD': {'primary_keys': ['IDENT']},
    'TFAIN': {'primary_keys': ['IDENT', 'JAHR', 'VON']},
    'THOC': {'primary_keys': ['IDENT', 'DATUM', 'FTGID']},
    'AUSP': {'primary_keys': ['MAFID', 'KLART', 'OBJEK', 'ATINN', 'ATZHL']},   # was read from the old Bronze lakehouse; Dab delivers AUSP to B1 (in the B1 swap list since Oct)
    'EINA': {'primary_keys': ['INFNR'], 'source_lh_key': 'B1_PROC_LH_ABFSS'},
    'EINE': {'primary_keys': ['INFNR', 'EKORG'], 'source_lh_key': 'B1_PROC_LH_ABFSS'},
    'EORD': {'primary_keys': ['MATNR', 'WERKS', 'ZEORD'], 'source_lh_key': 'B1_PROC_LH_ABFSS'},
    'T014': {'primary_keys': ['KKBER']},
    'KNKK': {'primary_keys': ['KUNNR', 'KKBER']},
    'SKA1': {'primary_keys': ['KTOPL', 'SAKNR']},
    'TBSL': {'primary_keys': ['BSCHL']},
    # ── New 23 Nov 2026: swap-list tables not loaded before (19) ──
    'LFBK': {'primary_keys': ['LIFNR', 'BANKS', 'BANKL', 'BANKN']},
    'LFC1': {'primary_keys': ['LIFNR', 'BUKRS', 'GJAHR']},
    'KNB1': {'primary_keys': ['KUNNR', 'BUKRS']},
    'KNBK': {'primary_keys': ['KUNNR', 'BANKS', 'BANKL', 'BANKN']},
    'KNB5': {'primary_keys': ['KUNNR', 'BUKRS', 'MABER']},
    'KNKA': {'primary_keys': ['KUNNR']},
    'KNVH': {'primary_keys': ['HITYP', 'KUNNR', 'VKORG', 'VTWEG', 'SPART', 'DATAB']},
    'KNVK': {'primary_keys': ['PARNR']},
    'KONA': {'primary_keys': ['KNUMA']},
    'KNC1': {'primary_keys': ['KUNNR', 'BUKRS', 'GJAHR']},
    'KNMT': {'primary_keys': ['VKORG', 'VTWEG', 'KUNNR', 'MATNR']},
    'ADCP': {'primary_keys': ['ADDRNUMBER', 'PERSNUMBER', 'DATE_FROM', 'NATION']},
    'ADR2': {'primary_keys': ['ADDRNUMBER', 'PERSNUMBER', 'DATE_FROM', 'CONSNUMBER']},
    'ADR3': {'primary_keys': ['ADDRNUMBER', 'PERSNUMBER', 'DATE_FROM', 'CONSNUMBER']},
    'ADR6': {'primary_keys': ['ADDRNUMBER', 'PERSNUMBER', 'DATE_FROM', 'CONSNUMBER']},
    'ADRCT': {'primary_keys': ['ADDRNUMBER', 'DATE_FROM', 'NATION', 'LANGU']},
    'ADRP': {'primary_keys': ['PERSNUMBER', 'DATE_FROM', 'NATION']},
    'ADRT': {'primary_keys': ['ADDRNUMBER', 'PERSNUMBER', 'DATE_FROM', 'CONSNUMBER', 'COMM_TYPE', 'LANGU']},
    'ZSDINDUSTRY_SEG3': {'primary_keys': ['LEVEL3']},   # key assumed from SEG0-2: confirm with Dab
}

MD_DESC_CONFIGS = {                   # 104 description tables (loaded with DESC_BRONZE_MODE)
    # ── MD texts (49) ──
    'MAKT': {'primary_keys': ['MATNR'], 'spras': True, 'text_col': 'MAKTX'},
    'T134T': {'primary_keys': ['MTART'], 'spras': True, 'text_col': 'MTBEZ'},
    'TSPAT': {'primary_keys': ['SPART'], 'spras': True, 'text_col': 'VTEXT'},
    'T179T': {'primary_keys': ['PRODH'], 'spras': True, 'text_col': 'VTEXT'},
    'TWEWT': {'primary_keys': ['EXTWG'], 'spras': True, 'text_col': 'EWBEZ'},
    'T149T': {'primary_keys': ['BWTTY'], 'spras': True, 'text_col': 'BTBEZ'},
    'T024D': {'primary_keys': ['WERKS', 'DISPO'], 'spras': False, 'text_col': 'DSNAM'},
    'T001L': {'primary_keys': ['WERKS', 'LGORT'], 'spras': False, 'text_col': 'LGOBE'},
    'TVKOT': {'primary_keys': ['VKORG'], 'spras': True, 'text_col': 'VTEXT'},
    'TVTWT': {'primary_keys': ['VTWEG'], 'spras': True, 'text_col': 'VTEXT'},
    'TVMST': {'primary_keys': ['VMSTA'], 'spras': True, 'text_col': 'VMSTB'},
    'TVM1T': {'primary_keys': ['MVGR1'], 'spras': True, 'text_col': 'BEZEI'},
    'TVM2T': {'primary_keys': ['MVGR2'], 'spras': True, 'text_col': 'BEZEI'},
    'TVM3T': {'primary_keys': ['MVGR3'], 'spras': True, 'text_col': 'BEZEI'},
    'TVM4T': {'primary_keys': ['MVGR4'], 'spras': True, 'text_col': 'BEZEI'},
    'TVM5T': {'primary_keys': ['MVGR5'], 'spras': True, 'text_col': 'BEZEI'},
    'T016T': {'primary_keys': ['BRSCH'], 'spras': True, 'text_col': 'BRTXT'},
    'TBRCT': {'primary_keys': ['BRACO'], 'spras': True, 'text_col': 'VTEXT'},
    'ZSDINDUSTRY_SEG0': {'primary_keys': ['LEVEL4']},
    'ZSDINDUSTRY_SEG1': {'primary_keys': ['LEVEL1'], 'spras': False, 'text_col': 'DESCRIPTION'},
    'ZSDINDUSTRY_SEG2': {'primary_keys': ['LEVEL2'], 'spras': False, 'text_col': 'DESCRIPTION'},
    'T151T': {'primary_keys': ['KDGRP'], 'spras': True, 'text_col': 'KTEXT'},
    'T171T': {'primary_keys': ['BZIRK'], 'spras': True, 'text_col': 'BZTXT'},
    'TVV1T': {'primary_keys': ['KVGR1'], 'spras': True, 'text_col': 'BEZEI'},
    'TVV2T': {'primary_keys': ['KVGR2'], 'spras': True, 'text_col': 'BEZEI'},
    'TVV3T': {'primary_keys': ['KVGR3'], 'spras': True, 'text_col': 'BEZEI'},
    'TVV4T': {'primary_keys': ['KVGR4'], 'spras': True, 'text_col': 'BEZEI'},
    'TVV5T': {'primary_keys': ['KVGR5'], 'spras': True, 'text_col': 'BEZEI'},
    'T077Y': {'primary_keys': ['KTOKK'], 'spras': True, 'text_col': 'TXT30'},
    'T052U': {'primary_keys': ['ZTERM'], 'spras': True, 'text_col': 'TEXT1'},
    'T005T': {'primary_keys': ['LAND1'], 'spras': True, 'text_col': 'LANDX'},
    'T005U': {'primary_keys': ['LAND1', 'BLAND'], 'spras': True, 'text_col': 'BEZEI'},
    'TVKBT': {'primary_keys': ['VKBUR'], 'spras': True, 'text_col': 'BEZEI'},
    'T001T': {'primary_keys': ['BUKRS'], 'spras': False, 'text_col': 'TXTNA'},
    'T161T': {'primary_keys': ['BSART'], 'spras': True, 'text_col': 'BATXT'},
    'T163C': {'primary_keys': ['BEWTP'], 'spras': True, 'text_col': 'BEWTL'},
    'T156HT': {'primary_keys': ['BWART'], 'spras': True, 'text_col': 'BTEXT'},
    'T16FT': {'primary_keys': ['FRGGR', 'FRGSX'], 'spras': True, 'text_col': 'FRGXT'},
    'T16FH': {'primary_keys': ['FRGGR'], 'spras': True, 'text_col': 'FRGGT'},
    'T16FE': {'primary_keys': ['FRGKE'], 'spras': True, 'text_col': 'FRGET'},
    'T163Y': {'primary_keys': ['PSTYP', 'SPRAS']},
    'ZMATGROUP': {'primary_keys': ['MATKL'], 'spras': False, 'text_col': 'ZMTFM'},
    'CRTX': {'primary_keys': ['OBJTY', 'OBJID', 'SPRAS'], 'spras': True, 'text_col': 'KTEXT'},
    'T148T': {'primary_keys': ['SOBKZ', 'SPRAS'], 'spras': True, 'text_col': 'SOTXT'},
    'EQKT': {'primary_keys': ['EQUNR'], 'spras': True, 'text_col': 'EQKTX'},
    'EQBS': {'primary_keys': ['EQUNR']},
    'CAWNT': {'primary_keys': ['ATINN', 'ATZHL', 'SPRAS', 'ADZHL'], 'spras': True, 'text_col': 'ATWTB'},
    'DD07T': {'primary_keys': ['DOMVALUE_L'], 'spras': False, 'text_col': 'DDTEXT'},
    'T460T': {'primary_keys': ['WERKS', 'SOBSL'], 'spras': True, 'text_col': 'LTEXT'},

    # ── texts MD loads for Sales (36) ──
    'TVAKT': {'primary_keys': ['AUART', 'SPRAS']},
    'TVAUT': {'primary_keys': ['AUGRU', 'SPRAS']},
    'TVFKT': {'primary_keys': ['FKART', 'SPRAS']},
    'T176T': {'primary_keys': ['BSARK', 'SPRAS']},
    'T014T': {'primary_keys': ['KKBER', 'SPRAS']},
    'T691T': {'primary_keys': ['KKBER', 'CTLPC', 'SPRAS']},
    'TVLVT': {'primary_keys': ['ABRVW', 'SPRAS']},
    'TVSBT': {'primary_keys': ['VSBED', 'SPRAS']},
    'TVSTT': {'primary_keys': ['VSTEL', 'SPRAS']},
    'TVFST': {'primary_keys': ['FAKSP', 'SPRAS']},
    'TVROT': {'primary_keys': ['ROUTE', 'SPRAS']},
    'TMVFT': {'primary_keys': ['MTVFP', 'SPRAS']},
    'T178T': {'primary_keys': ['KONDM', 'SPRAS']},
    'CEPCT': {'primary_keys': ['KOKRS', 'PRCTR', 'DATBI', 'SPRAS']},
    'T006A': {'primary_keys': ['MSEHI', 'SPRAS']},
    'T459W': {'primary_keys': ['BEDAE', 'SPRAS']},
    'TVEPT': {'primary_keys': ['ETTYP', 'SPRAS']},
    'TVLST': {'primary_keys': ['LIFSP', 'SPRAS']},
    'TVLKT': {'primary_keys': ['LFART', 'SPRAS']},
    'TINCT': {'primary_keys': ['INCO1', 'SPRAS']},
    'TPRIT': {'primary_keys': ['LPRIO', 'SPRAS']},
    'TTGRT': {'primary_keys': ['TRAGR', 'SPRAS']},
    'T300T': {'primary_keys': ['LGNUM', 'SPRAS']},
    'TTSGT': {'primary_keys': ['TRSPG', 'SPRAS']},
    'T30CT': {'primary_keys': ['LGNUM', 'LGBZO', 'SPRAS']},
    'T173T': {'primary_keys': ['VSART', 'SPRAS']},
    'TVAPT': {'primary_keys': ['PSTYV', 'SPRAS']},
    'TLGRT': {'primary_keys': ['LADGR', 'SPRAS']},
    'T023T': {'primary_keys': ['MATKL', 'SPRAS']},
    'TGSBT': {'primary_keys': ['GSBER', 'SPRAS']},
    'TVEGRT': {'primary_keys': ['MAGRV', 'SPRAS']},
    'TMFGT': {'primary_keys': ['MFRGR', 'SPRAS']},
    'MDLV': {'primary_keys': ['BERID']},
    'T024B': {'primary_keys': ['KKBER', 'SBGRP']},
    'TDG42': {'primary_keys': ['PROFL', 'DGSPRAS']},
    'TCURW': {'primary_keys': ['KURST', 'SPRAS']},

    # ── texts MD loads for Audit (8) ──
    'SKAT': {'primary_keys': ['KTOPL', 'SAKNR', 'SPRAS']},
    'TBSLT': {'primary_keys': ['BSCHL', 'SPRAS']},
    'T003T': {'primary_keys': ['BLART', 'SPRAS']},
    'T030': {'primary_keys': ['KTOPL', 'KTOSL', 'BWMOD', 'KOMOK', 'BKLAS']},
    'T008T': {'primary_keys': ['ZAHLS', 'SPRAS']},
    'CSKT': {'primary_keys': ['KOKRS', 'KOSTL', 'DATBI', 'SPRAS'], 'source_lh_key': 'B1_AUDIT_LH_ABFSS'},
    'CEPC': {'primary_keys': ['KOKRS', 'PRCTR', 'DATBI']},
    'TKT05': {'primary_keys': ['KOSAR', 'SPRAS']},

    # ── texts MD loads for Customer Service (11) ──
    'TQ80_T': {'primary_keys': ['QMART', 'SPRAS']},
    'TQ80': {'primary_keys': ['QMART']},
    'T356A_T': {'primary_keys': ['ARTPR', 'SPRAS']},
    'QPCT': {'primary_keys': ['KATALOGART', 'CODEGRUPPE', 'CODE', 'VERSION']},
    'QPGT': {'primary_keys': ['KATALOGART', 'CODEGRUPPE']},
    'T003P': {'primary_keys': ['AUART', 'SPRAS']},
    'TCURT': {'primary_keys': ['WAERS', 'SPRAS']},
    'TJ02T': {'primary_keys': ['ISTAT', 'SPRAS']},
    'CABN': {'primary_keys': ['ATINN', 'ADZHL']},
    'CABNT': {'primary_keys': ['ATINN', 'SPRAS', 'ADZHL']},
    'CAWN': {'primary_keys': ['ATINN', 'ATZHL', 'ADZHL']},
}